# Properties

A *property* is a Python mechanism that controls read, write and delete access to an attribute, while keeping a simple access syntax (without parentheses).

In these practical exercises, we will learn how to use them.

## Defining a property: the getter

Write a `Circle` class that has:

- a "private" attribute `_radius`, initialized in the constructor from a `radius` argument
- a `radius` property that returns the value of `_radius`

The property must be readable through `instance.radius`, without parentheses.

In [ ]:
class Circle:
  def __init__(self, radius):
    self._radius = radius

  # Your code here


c = Circle(5)
print(c.radius)  # Should be 5

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self._radius = radius

  @property
  def radius(self):
    return self._radius


c = Circle(5)
print(c.radius)  # Should be 5

## Property syntax

The `@property` syntax is syntactic sugar. Without it, how can you get the same result, using the built-in `property` function directly?

In [ ]:
# Your code here

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self._radius = radius

  def get_radius(self):
    return self._radius

  radius = property(get_radius)


c = Circle(5)
print(c.radius)  # Should be 5

## Adding a setter

Take the `Circle` class again and add a setter to the `radius` property that checks that the given value is strictly positive. If it is not, a `ValueError` must be raised.

Also change the constructor so that it uses `self.radius = radius` (and not `self._radius = radius`), so that the validation also applies when the object is created.

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  # Your code here


c = Circle(5)
print(c.radius)  # Should be 5

c.radius = 10
print(c.radius)  # Should be 10

c.radius = -1  # Should raise a ValueError

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value


c = Circle(5)
print(c.radius)  # 5

c.radius = 10
print(c.radius)  # 10

try:
  c.radius = -1
except ValueError as e:
  print(f"Error: {e}")

## Adding a deleter

Add a deleter to the `radius` property that prints the message `"Deleting radius"` and then deletes the `_radius` attribute from the instance.

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value

  # Your code here


c = Circle(5)
del c.radius
print(hasattr(c, "_radius"))  # Should be False

### Solution

In [ ]:
class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value

  @radius.deleter
  def radius(self):
    print("Deleting radius")
    del self._radius


c = Circle(5)
del c.radius
print(hasattr(c, "_radius"))  # Should be False

## Read-only property

Add to the `Circle` class a read-only `area` property (without a setter) that computes the area of the disk from `radius` (use `math.pi`).

In [ ]:
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value

  # Your code here


c = Circle(5)
print(c.area)  # Should be about 78.53981633974483

What happens if you run `c.area = 10`? Try it before reading the solution.

*Your answer*

### Solution

A property without a setter is read-only: any assignment raises an `AttributeError`, since no setter is defined to intercept the write.

Internally, `property` is a *data descriptor*: it always defines `__set__` (which, without a setter, just raises the error), which gives it priority over the instance's `__dict__`.

In [ ]:
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value

  @property
  def area(self):
    return math.pi * self._radius ** 2


c = Circle(5)
print(c.area)

try:
  c.area = 10
except AttributeError as e:
  print(f"Error: {e}")

## Caching with `functools.cached_property`

Computing `area` is trivial here, but imagine it were an expensive computation (a network request, a complex calculation...). It would be a waste to redo it on every access if `radius` does not change in between.

Use [`functools.cached_property`](https://docs.python.org/3/library/functools.html#functools.cached_property) to compute `area` only once, and check it by counting how many times the computation message is displayed.

In [ ]:
import functools
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value

  # Your code here: change the area property
  @property
  def area(self):
    print("Computing the area...")
    return math.pi * self._radius ** 2


c = Circle(5)
print(c.area)
print(c.area)  # For now, "Computing the area..." is displayed twice: it should be displayed only once

### Solution

In [ ]:
import functools
import math


class Circle:
  def __init__(self, radius):
    self.radius = radius

  @property
  def radius(self):
    return self._radius

  @radius.setter
  def radius(self, value):
    if value <= 0:
      raise ValueError("radius must be positive")
    self._radius = value

  @functools.cached_property
  def area(self):
    print("Computing the area...")
    return math.pi * self._radius ** 2


c = Circle(5)
print(c.area)
print(c.area)  # The message is displayed only once

Beware: `cached_property` requires instances to have a `__dict__` (the default, unless the class defines `__slots__` without `"__dict__"`), and the cache is not invalidated automatically if `radius` changes later: you would have to handle it manually, for instance by deleting `self.__dict__["area"]` in the `radius` setter.